# LivePortrait GStreamer Plugin Tutorial

This notebook demonstrates how to build and use the `gst-liveportrait` plugin for real-time head reenactment.

## 1. Prerequisites

Ensure you have the Docker image built and the TensorRT engines exported as described in the `README.md`.

## 2. Compile the Plugin

Run the following command to compile the C++ plugin inside the Docker container.

In [ ]:
!docker run --rm -v $(pwd)/..:/workspace -w /workspace gst-liveportrait-env bash -c "mkdir -p build && cd build && cmake .. && make -j$(nproc)"

## 3. Run with GStreamer Command Line

The following command runs the reenactment pipeline using `gst-launch-1.0`.

In [ ]:
!docker run --rm --gpus all -v $(pwd)/..:/workspace -w /workspace gst-liveportrait-env bash -c "\
    GST_PLUGIN_PATH=./build gst-launch-1.0 \
    filesrc location=assets/video_example.mp4 ! \
    decodebin ! videoconvert ! \
    videocrop left=280 right=280 ! \
    videoscale ! video/x-raw,width=512,height=512,format=RGB ! \
    liveportrait config-path=./checkpoints source-image=assets/test_image.jpg ! \
    videoconvert ! x264enc ! mp4mux ! filesink location=outputs/tutorial_output.mp4"

## 4. Run using the Python Wrapper

You can also use the `LivePortraitProcess` class for easier integration.

In [ ]:
import sys
sys.path.append("..")
from liveportrait_process import LivePortraitProcess

# Note: This code needs to run inside the Docker container to access GStreamer and the GPU
# processor = LivePortraitProcess(plugin_path="../build")
# processor.process(
#     input_video="../assets/video_example.mp4",
#     output_video="../outputs/python_output.mp4",
#     source_image="../assets/test_image.jpg",
#     config_path="../checkpoints"
# )